# Revenue Analysis Case Study
#### Data Preparation, Revenue Trend Analysis & Root Cause Investigation

**Prepared By:** Deepak Pathak   
**Date:** May 2026

# 1. Project Overview

## Business Context

The business has experienced a noticeable decline in revenue performance. The objective of this analysis is to investigate revenue trends, identify when and where the decline began, determine the key contributing factors, and provide actionable recommendations.

This project includes:

- Data Preparation
- Data Quality Assessment
- Revenue Trend Analysis
- Root Cause Investigation
- Dashboard Development in Power BI
- Business Recommendations

# 2. Project Objectives

The primary objectives of this analysis are:

- Assess and improve data quality across all provided datasets.
- Analyze revenue performance over time.
- Identify revenue trends by channel, region, and product category.
- Investigate the root causes behind the revenue decline.
- Examine order volume, cancellation rates, return rates, discount levels, and customer behavior.
- Build an interactive Power BI dashboard for ongoing business monitoring.
- Provide data-driven recommendations to improve revenue performance.

# 3. Datasets Used

The following datasets were provided for analysis:

### Orders Dataset
Contains order-level transaction information including revenue, order status, sales channel, and order dates.

### Order Items Dataset
Contains product-level transaction details including quantity, unit price, category, and line-level revenue.

### Customers Dataset
Contains customer information including customer identifiers and regional attributes.

### Returns Dataset
Contains return transactions and return reasons associated with completed orders.

# 4. Tools & Technologies Used

### Programming & Analysis
- Python
- Pandas
- NumPy

### Data Visualization
- Matplotlib
- Seaborn

### Business Intelligence
- Power BI

### Development Environment
- Jupyter Notebook

# 5. Data Quality & Preparation Approach

The data preparation process involved:

- Data inspection and structure validation
- Missing value assessment
- Data type standardization
- Date format conversion
- Revenue validation checks
- Duplicate record identification
- Dataset relationship validation
- Data model preparation for Power BI reporting

# 6. Methodology

The analysis was conducted in three stages:

## Phase 1 – Data Preparation
Identification and resolution of data quality issues across all datasets.

## Phase 2 – Revenue Trend Analysis
Evaluation of revenue performance by:
- Month
- Region
- Sales Channel
- Product Category

## Phase 3 – Root Cause Investigation
Analysis of:
- Order Volume
- Cancellation Rates
- Return Rates
- Discount Trends
- Customer Purchase Behavior
- Return Reasons

# 7. Expected Deliverables

The final deliverables of this project include:

- Cleaned and validated datasets
- Revenue trend analysis
- Root cause investigation
- Interactive Power BI dashboard
- Business recommendations supported by data

# 8. Import Required Libraries
The following libraries are used for data manipulation, analysis, and visualization throughout the project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)

# 9. Load Datasets

In [ ]:
orders = pd.read_csv('orders.csv')
order_items = pd.read_csv('order_items.csv')
customers = pd.read_csv('customers.csv')
returns = pd.read_csv('returns.csv')

# 10. Data Inspection

In [ ]:
print(orders.head())
print(order_items.head())
print(customers.head())
print(returns.head())

In [ ]:
print(orders.info())
print(order_items.info())
print(customers.info())
print(returns.info())

# 11. Data Cleaning & Preparation

### Check Missing Values

In [ ]:
print("Orders Missing Values")
print(orders.isnull().sum())

print("\nOrder Items Missing Values")
print(order_items.isnull().sum())

print("\nCustomers Missing Values")
print(customers.isnull().sum())

print("\nReturns Missing Values")
print(returns.isnull().sum())

### Check Duplicate Records

In [ ]:
print("Orders duplicates:", orders.duplicated().sum())
print("Order Items duplicates:", order_items.duplicated().sum())
print("Customers duplicates:", customers.duplicated().sum())
print("Returns duplicates:", returns.duplicated().sum())

### Remove Duplicate Records

In [ ]:
orders.drop_duplicates(inplace=True)
order_items.drop_duplicates(inplace=True)
customers.drop_duplicates(inplace=True)
returns.drop_duplicates(inplace=True)

### Convert Date Columns

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'],format='mixed',dayfirst=True)

In [ ]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])

In [ ]:
returns['return_date'] = pd.to_datetime(returns['return_date'])

### Standardize Text Columns

In [ ]:
orders['status'] = (orders['status'].str.lower().str.strip())
orders['channel'] = (orders['channel'].str.strip())
customers['region'] = (customers['region'].str.strip())
customers['customer_segment'] = (customers['customer_segment'].str.strip())
order_items['category'] = (order_items['category'].str.strip())
returns['reason'] = (returns['reason'].str.strip())


### Validate Unique Values

In [ ]:
print("Order Status Values")
print(orders['status'].unique())

print("\nSales Channels")
print(orders['channel'].unique())

print("\nRegions")
print(customers['region'].unique())

print("\nCustomer Segments")
print(customers['customer_segment'].unique())

print("\nProduct Categories")
print(order_items['category'].unique())

### Handle Missing Discounts

In [ ]:
orders['discount_applied'] = orders['discount_applied'].fillna(0)

### Revenue Validation

In [ ]:
order_items['line_total'] = order_items['unit_price'] * order_items['quantity']

In [ ]:
calculated_totals = order_items.groupby('order_id')['line_total'].sum().reset_index()

In [ ]:
calculated_totals.rename(columns = {'line_total': 'calculated_total'},inplace=True)

In [ ]:
calculated_totals

In [ ]:
orders = orders.merge(calculated_totals,on='order_id',how='left')

In [ ]:
orders['revenue_difference'] = (orders['total_amount'] -orders['calculated_total'])

In [ ]:
orders

In [ ]:
print(orders[['order_id','total_amount','calculated_total','revenue_difference']].head(5))

In [ ]:
orders['expected_total'] = (orders['calculated_total'] * (1 -orders['discount_applied'] / 100))

In [ ]:
orders

In [ ]:
orders['revenue_variance'] = (orders['total_amount'] -orders['expected_total'])

In [ ]:
orders

### Check Null Dates after Conversion

In [ ]:
print("Invalid Order Dates:",orders['order_date'].isnull().sum())
print("Invalid Signup Dates:",customers['signup_date'].isnull().sum())
print("Invalid Return Dates:",returns['return_date'].isnull().sum())

### Create Additional BI Columns

In [ ]:
#month-year column
orders['month_year'] = (orders['order_date'].dt.strftime('%Y-%m'))

In [ ]:
#cancellation flag
orders['is_cancelled'] = np.where(orders['status'] == 'cancelled',1,0)

In [ ]:
#completed order flag
orders['is_completed'] = np.where(orders['status'] == 'completed',1,0)

### Merge Datasets for Analysis

In [ ]:
merged_data = orders.merge(customers,on='customer_id',how='left')
merged_data = merged_data.merge(order_items,on='order_id',how='left')

In [ ]:
print(merged_data.shape)
print(merged_data.head())
print(merged_data.isnull().sum())

### Export Cleaned Files

In [ ]:
orders.to_csv('clean_orders.csv',index=False)

order_items.to_csv('clean_order_items.csv',index=False)

customers.to_csv('clean_customers.csv',index=False)

returns.to_csv('clean_returns.csv',index=False)

merged_data.to_csv('merged_data.csv',index=False)
print("Cleaned datasets exported successfully.")

### Professional Data Cleaning Summary
Cleaning Steps Performed:
- Removed duplicate records from all datasets
- Standardized text fields
- Converted date columns into datetime format
- Handled missing discount values
- Created product-level revenue column (line_total)
- Validated order revenue against item-level totals
- Created additional analytical columns
- Prepared Power BI-ready cleaned datasets

## Insight
The raw transactional datasets were first validated for data quality issues including duplicates, missing values, inconsistent text formatting, and invalid date formats. Additional calculated fields such as product-level revenue and cancellation indicators were created to support downstream business analysis and Power BI dashboarding.

# 12. Revenue Trend Analysis

### 1. Monthly Revenue Trend Analysis

In [ ]:
monthly_revenue = orders.groupby('month_year')['total_amount'].sum().reset_index()
monthly_revenue

In [ ]:
#Visualization
plt.figure(figsize = (14,6))

plt.plot(monthly_revenue['month_year'],
         monthly_revenue['total_amount'],marker = 'o')
plt.xticks(rotation = 45)
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.grid(True)
plt.show()

#### Insight
Revenue remained relatively stable during the early business periods but began declining significantly from mid-2024 onward. The trend indicates a sustained reduction in business performance rather than a short-term fluctuation.

#### Business Finding
The sharpest revenue decline appears after July 2024, suggesting operational or demand-related disruptions beginning during this period.

### 2. Revenue Analysis by Sales Channel

In [ ]:
channel_revenue = orders.groupby('channel')['total_amount'].sum().sort_values(ascending = False).reset_index()
channel_revenue

In [ ]:
# Visualization
plt.figure(figsize=(10,6))

sns.barplot(data=channel_revenue,x='channel',y='total_amount')

plt.title('Revenue by Sales Channel')
plt.xlabel('Channel')
plt.ylabel('Revenue')

plt.xticks(rotation=30)

plt.show()

#### Monthly Channel Trend Analysis

In [ ]:
channel_monthly = (orders.groupby(['month_year','channel'])['total_amount'].sum().reset_index())
channel_monthly

In [ ]:
#visualization
plt.figure(figsize=(15,7))

sns.lineplot(
    data=channel_monthly,
    x='month_year',
    y='total_amount',
    hue='channel',
    marker='o')

plt.xticks(rotation=45)
plt.title('Monthly Revenue by Channel')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.grid(True)
plt.show()

#### Business Insight
Certain channels experienced steeper declines than others, indicating that the revenue downturn may be partially driven by weakening performance in specific acquisition channels.

### 3. Revenue Analysis by Region

In [ ]:
region_analysis = orders.merge(customers,on='customer_id',how='left')

In [ ]:
region_revenue = (region_analysis.groupby('region')['total_amount'].sum().sort_values(ascending=False).reset_index())
region_revenue

In [ ]:
#Visualization
plt.figure(figsize=(10,6))

sns.barplot(
    data=region_revenue,
    x='region',
    y='total_amount')

plt.title('Revenue by Region')
plt.xlabel('Region')
plt.ylabel('Revenue')

plt.xticks(rotation=30)
plt.show()

#### Insight
Revenue performance differs across regions, suggesting uneven market performance and possible region-specific operational or demand challenges.

#### Monthly Regional Trend

In [ ]:
region_monthly = (region_analysis.groupby(['month_year','region'])['total_amount'].sum().reset_index())

In [ ]:
#Visualization
# Plot
plt.figure(figsize=(15,7))
sns.lineplot(
    data=region_monthly,
    x='month_year',
    y='total_amount',
    hue='region',
    marker='o')

plt.xticks(rotation=45)
plt.title('Monthly Revenue by Region')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.grid(True)
plt.show()

#### Business Finding
Some regions experienced a sharper decline after mid-2024, indicating that the downturn was not evenly distributed across all markets.

### 4. Revenue Analysis by Product Category

In [ ]:
category_analysis = orders.merge(order_items,on='order_id',how='left')

In [ ]:
category_revenue = (category_analysis.groupby('category')['line_total'].sum().sort_values(ascending=False).reset_index())
category_revenue

In [ ]:
#Visualization
plt.figure(figsize=(10,6))

sns.barplot(data=category_revenue,x='category',y='line_total')
plt.title('Revenue by Product Category')
plt.xlabel('Category')
plt.ylabel('Revenue')
plt.xticks(rotation=30)
plt.show()

#### Insight
Product category contribution varies significantly, indicating that some categories are more resilient while others may be driving the revenue decline.

Specific product categories experienced earlier and steeper declines, suggesting that changing customer demand or inventory/product issues may have contributed to overall revenue deterioration.

## Summary
Revenue trend analysis revealed that the business began experiencing a sustained decline beginning around mid-2024. The decline was observable across multiple dimensions including sales channels, regions, and product categories, indicating broader operational and demand-side challenges rather than isolated short-term fluctuations.

# 13. Root Cause Investigation

### 1. Order Volume Analysis

In [ ]:
monthly_orders = (orders.groupby('month_year')['order_id'].count().reset_index())
monthly_orders

In [ ]:
# Visualization
plt.figure(figsize=(14,6))

plt.plot(
    monthly_orders['month_year'],
    monthly_orders['order_id'],
    marker='o')

plt.xticks(rotation=45)
plt.title('Monthly Order Volume Trend')
plt.xlabel('Month')
plt.ylabel('Number of Orders')
plt.grid(True)
plt.show()

#### Insight
- Order volume declined significantly during the same period as revenue deterioration, indicating that falling customer demand or acquisition challenges were major contributors to the decline.
- The reduction in revenue strongly correlates with declining order counts, suggesting that reduced transaction activity was a primary driver of business deterioration.

### 2. Cancellation Rate Analysis

In [ ]:
monthly_cancellation = (
    orders.groupby('month_year').
    agg(total_orders=('order_id', 'count'),cancelled_orders=('is_cancelled', 'sum')).reset_index())

In [ ]:
monthly_cancellation

In [ ]:
monthly_cancellation['cancellation_rate'] = round(monthly_cancellation['cancelled_orders']/monthly_cancellation['total_orders'] * 100,2)

In [ ]:
monthly_cancellation

In [ ]:
# Visualization
plt.figure(figsize=(14,6))

plt.plot(
    monthly_cancellation['month_year'],
    monthly_cancellation['cancellation_rate'],
    marker='o')

plt.xticks(rotation=45)
plt.title('Monthly Cancellation Rate')
plt.xlabel('Month')
plt.ylabel('Cancellation Rate (%)')
plt.grid(True)
plt.show()

#### Insight
- Cancellation rates increased noticeably during the revenue decline period, indicating potential operational inefficiencies such as inventory shortages, fulfillment issues, or delayed delivery commitments.
- Rising cancellation rates likely contributed directly to revenue loss and may indicate weakening operational performance.

### 3. Return Rate Analysis

In [ ]:
#merge orders with returns
returns_analysis = orders.merge(returns,on='order_id',how='left')

In [ ]:
#create return flag
returns_analysis['is_returned'] = np.where(returns_analysis['return_id'].notnull(),1,0)

In [ ]:
monthly_returns = (returns_analysis.groupby('month_year').agg(total_orders=('order_id', 'count'),
                                               returned_orders=('is_returned', 'sum')).reset_index())
monthly_returns

In [ ]:
#monthly return rate
monthly_returns['return_rate'] = round((monthly_returns['returned_orders'] /monthly_returns['total_orders']) * 100,2)

In [ ]:
# Visualization
plt.figure(figsize=(14,6))

plt.plot(
    monthly_returns['month_year'],
    monthly_returns['return_rate'],
    marker='o')
plt.xticks(rotation=45)
plt.title('Monthly Return Rate')
plt.xlabel('Month')
plt.ylabel('Return Rate (%)')
plt.grid(True)
plt.show()

#### Insight
Increasing return rates suggest growing customer dissatisfaction, product quality issues, or delivery-related problems, all of which negatively impact realized revenue.

### Return Reason Analysis

In [ ]:
return_reason = (returns['reason'].value_counts().reset_index())
return_reason

In [ ]:
# Visualization
plt.figure(figsize=(10,6))
sns.barplot(
    data=return_reason,
    x='reason',
    y='count')

plt.title('Return Reasons')
plt.xlabel('Reason')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()

#### Insight
Return reason analysis highlights operational pain points such as damaged products, incorrect items, or delayed deliveries that may be contributing to declining customer satisfaction.

## 4. Discount Level Analysis

In [ ]:
#convert discount to percentage
orders['discount_percentage'] = (orders['discount_applied'] * 100)

In [ ]:
monthly_discount = (orders.groupby('month_year')['discount_percentage'].mean().reset_index())
monthly_discount

In [ ]:
# Visualization
plt.figure(figsize=(14,6))

plt.plot(
    monthly_discount['month_year'],
    monthly_discount['discount_percentage'],
    marker='o')

plt.xticks(rotation=45)
plt.title('Average Monthly Discount Trend')
plt.xlabel('Month')
plt.ylabel('Average Discount (%)')
plt.grid(True)
plt.show()

#### Insight
- Discount values were originally stored in decimal format and were converted into percentage representation to improve interpretability and business readability during analysis and dashboard reporting.
- Discount levels increased during the decline period, indicating attempts to stimulate demand through pricing incentives.
- Despite higher discounts, revenue continued to decline, suggesting that pricing incentives alone were insufficient to offset broader operational or demand-side issues.

## 5. New vs Repeat Customer Analysis

In [ ]:
customer_order_count = (orders.groupby('customer_id')['order_id'].count().reset_index())
customer_order_count

In [ ]:
customer_order_count.rename(columns={'order_id': 'total_orders'},inplace=True)

In [ ]:
#creating repeat customer flag
customer_order_count['is_repeat_customer'] = np.where(customer_order_count['total_orders'] > 1,1,0)
customer_order_count

In [ ]:
customers = customers.merge(customer_order_count[['customer_id','is_repeat_customer']],on='customer_id',how='left')

In [ ]:
customer_mix = (customers['is_repeat_customer'].value_counts().reset_index())

customer_mix.columns = ['customer_type','count']

# Replace values
customer_mix['customer_type'] = (customer_mix['customer_type'].replace({0: 'New Customer',1: 'Repeat Customer'}))

print(customer_mix)

In [ ]:
#Visualization
plt.figure(figsize=(8,6))

sns.barplot(
    data=customer_mix,
    x='customer_type',
    y='count')

plt.title('New vs Repeat Customers')
plt.xlabel('Customer Type')
plt.ylabel('Customer Count')
plt.show()

#### Insight
The original repeat customer field was unusable because it contained null values. Repeat customer behavior was therefore derived using transactional order history, where a customer’s first purchase was classified as a new customer order and subsequent purchases were classified as repeat customer orders.

# 14. Key Findings

Based on the data preparation, revenue trend analysis, and root cause investigation, the following key findings were identified:

### Revenue Performance
- Revenue remained relatively stable throughout 2023 and the first half of 2024.
- A noticeable decline in revenue began during Jul–Aug 2024 and continued thereafter.
- The decline was observed across multiple business dimensions, including regions, channels, and product categories.

### Order Volume
- Order volume decreased significantly during the revenue decline period.
- The reduction in orders contributed directly to lower overall revenue performance.

### Discount Analysis
- Average discount levels increased during the decline period.
- Despite higher discounts, revenue continued to decline, suggesting that promotional strategies alone were insufficient to maintain growth.

### Cancellation Analysis
- Cancellation rates increased substantially during the period of declining revenue.
- This indicates potential operational, fulfillment, or customer experience challenges.

### Return Analysis
- Return rates increased significantly during the same period.
- The rise in returns negatively impacted overall business performance and customer satisfaction.

### Return Reasons
- Late delivery and damaged products emerged as the most common return reasons.
- These issues suggest opportunities for improvement in logistics and quality-control processes.

### Customer Behavior
- Customer purchasing patterns indicated a strong dependence on repeat purchases.
- Maintaining customer satisfaction and retention appears critical for future revenue growth.

# 15. Recommendations

Based on the findings, the following recommendations are proposed:

1. Improve fulfillment and delivery operations to reduce cancellations and delivery-related returns.

2. Strengthen product quality-control processes to minimize damaged-product returns.

3. Review discount effectiveness and focus on customer experience improvements rather than relying primarily on promotional pricing.

4. Monitor cancellation rates, return rates, and customer retention metrics through an ongoing Power BI dashboard.

# 16. Conclusion

The analysis identified a significant revenue decline beginning in mid-2024. While discount levels increased, rising cancellation rates, return rates, and operational issues such as late deliveries and damaged products appear to be the primary contributors to declining performance.

Addressing these operational challenges and improving customer experience will be critical to restoring sustainable revenue growth.